# Cortex Agent Multi-Tenancy: Hands-On Lab

**Duration:** 30-45 minutes  
**Scenario:** You're building a multi-tenant sales analytics agent that serves 4 different companies. Each company should only see their own data, and within each company, users have different access levels (full/summary/restricted) that control which columns they can see.

**What you'll do:**
1. Explore the multi-tenant data model and entitlements table
2. Understand and test row access policies (tenant isolation)
3. Understand and test column masking policies (access levels)
4. Call a Cortex Agent with tenant context and see automatic row filtering
5. Observe column masking through the agent based on access level
6. Add a new user with zero DDL and see instant access
7. Review the audit trail for request attribution

**Prerequisites:** Run `setup.sql` before starting this notebook. It creates the database, tables, policies, semantic view, and agent.

**Key Concept:** Multi-tenancy in Cortex Agents uses *session attributes* passed at invocation time — no Snowflake accounts needed for end users. Row access policies and masking policies enforce isolation automatically.

In [ ]:
# Connection setup — works in both Snowsight notebooks and local Jupyter
import os
import json

try:
    # Snowsight notebook: session already exists
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    # Local Jupyter: create session from environment or connection config
    from snowflake.snowpark import Session
    connection_params = {
        "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
        "user": os.environ.get("SNOWFLAKE_USER"),
        "password": os.environ.get("SNOWFLAKE_PASSWORD"),
        "role": os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN"),
        "warehouse": "MULTI_TENANCY_WH",
        "database": "MULTI_TENANCY_LAB",
        "schema": "PUBLIC",
    }
    if not all([connection_params["account"], connection_params["user"], connection_params["password"]]):
        raise ValueError(
            "Missing required environment variables. "
            "Set SNOWFLAKE_ACCOUNT, SNOWFLAKE_USER, and SNOWFLAKE_PASSWORD."
        )
    session = Session.builder.configs(connection_params).create()

# Set context
session.sql("USE DATABASE MULTI_TENANCY_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE MULTI_TENANCY_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Database: {session.sql('SELECT CURRENT_DATABASE()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

In [ ]:
# Verify setup ran successfully
tables = session.sql("SHOW TABLES IN SCHEMA MULTI_TENANCY_LAB.PUBLIC").to_pandas()
print(f"Tables found: {len(tables)}")
print(tables[['"name"']].to_string(index=False))

sv = session.sql("SHOW SEMANTIC VIEWS IN SCHEMA MULTI_TENANCY_LAB.PUBLIC").to_pandas()
print(f"\nSemantic views: {len(sv)}")
print(sv[['"name"']].to_string(index=False))

---
## Section 2: Explore the Data

Before we apply policies, let's look at the raw data to understand what we're protecting. The `SALES_DATA` table contains transactions from 4 tenants, and `USER_ENTITLEMENTS` maps external users to tenants and access levels.

In [ ]:
# Overview of SALES_DATA — row counts per tenant
df = session.sql("""
    SELECT TENANT_ID, COUNT(*) AS ROW_COUNT, 
           SUM(AMOUNT) AS TOTAL_REVENUE,
           COUNT(DISTINCT PRODUCT) AS DISTINCT_PRODUCTS
    FROM SALES_DATA
    GROUP BY TENANT_ID
    ORDER BY TENANT_ID
""").to_pandas()
print("=== Sales Data by Tenant ===")
print(df.to_string(index=False))

In [ ]:
# Sample rows from SALES_DATA (note: customer names and emails are visible to us)
df = session.sql("""
    SELECT SALE_ID, TENANT_ID, REGION, PRODUCT, AMOUNT, SALE_DATE, 
           CUSTOMER_NAME, CUSTOMER_EMAIL
    FROM SALES_DATA
    LIMIT 10
""").to_pandas()
print("=== Sample Sales Data (no policies active for owner role) ===")
print(df.to_string(index=False))

In [ ]:
# USER_ENTITLEMENTS — the access control matrix
df = session.sql("""
    SELECT * FROM USER_ENTITLEMENTS ORDER BY TENANT_ID, ACCESS_LEVEL
""").to_pandas()
print("=== User Entitlements (Access Matrix) ===")
print(df.to_string(index=False))
print(f"\nTotal users: {len(df)}")
print(f"\nAccess level distribution:")
print(df['ACCESS_LEVEL'].value_counts().to_string())

---
## Section 3: Understand the Policies

Three policies enforce multi-tenancy:

| Policy | Type | What it does |
|--------|------|-------------|
| `RAP_TENANT_FILTER` | Row Access Policy | Filters rows so each tenant only sees their own data |
| `MASK_CUSTOMER_EMAIL` | Masking Policy | Hides email unless access_level = 'full' |
| `MASK_CUSTOMER_NAME` | Masking Policy | Hides name unless access_level IN ('full', 'summary') |

The key mechanism is `SYS_CONTEXT('SNOWFLAKE$SESSION_ATTRIBUTES', '<key>')` — this reads values passed at session/request time without requiring Snowflake user accounts.

In [ ]:
# View the row access policy definition
df = session.sql("""
    SELECT POLICY_NAME, POLICY_BODY 
    FROM INFORMATION_SCHEMA.ROW_ACCESS_POLICIES
    WHERE POLICY_NAME = 'RAP_TENANT_FILTER'
""").to_pandas()
print("=== Row Access Policy: RAP_TENANT_FILTER ===")
print(df['POLICY_BODY'].iloc[0] if len(df) > 0 else "Policy not found — check INFORMATION_SCHEMA access")

In [ ]:
# View masking policy definitions
df = session.sql("""
    SELECT POLICY_NAME, POLICY_BODY 
    FROM INFORMATION_SCHEMA.MASKING_POLICIES
    WHERE POLICY_NAME IN ('MASK_CUSTOMER_EMAIL', 'MASK_CUSTOMER_NAME')
    ORDER BY POLICY_NAME
""").to_pandas()
print("=== Masking Policies ===")
for _, row in df.iterrows():
    print(f"\n--- {row['POLICY_NAME']} ---")
    print(row['POLICY_BODY'])

### How SYS_CONTEXT Works

```sql
SYS_CONTEXT('SNOWFLAKE$SESSION_ATTRIBUTES', 'tenant_id')
```

This function reads a named attribute from the current session. In production, these attributes are set by your application layer (IdP integration, API gateway, etc.) when the external user authenticates. For testing, we can set them manually with:

```sql
ALTER SESSION SET SNOWFLAKE$SESSION_ATTRIBUTES = '{"tenant_id": "acme_corp", "user_id": "user_acme_admin"}';
```

The row access policy and masking policies both read from these attributes to enforce access control — the Cortex Agent inherits whatever session attributes are active.

---
## Section 4: Test Row Access Policy Manually

Let's simulate being different tenants by setting session attributes. The row access policy will automatically filter `SALES_DATA` to show only the matching tenant's rows.

In [ ]:
# Set session as acme_corp tenant
session.sql("""
    ALTER SESSION SET SNOWFLAKE$SESSION_ATTRIBUTES = 
    '{"tenant_id": "acme_corp", "user_id": "user_acme_admin"}'
""").collect()

df = session.sql("""
    SELECT TENANT_ID, COUNT(*) AS ROW_COUNT, SUM(AMOUNT) AS TOTAL_REVENUE
    FROM SALES_DATA
    GROUP BY TENANT_ID
""").to_pandas()
print("=== As acme_corp: Only acme_corp rows visible ===")
print(df.to_string(index=False))
print(f"\nTotal rows visible: {df['ROW_COUNT'].sum()}")

In [ ]:
# Switch to globex_inc — completely different data
session.sql("""
    ALTER SESSION SET SNOWFLAKE$SESSION_ATTRIBUTES = 
    '{"tenant_id": "globex_inc", "user_id": "user_globex_admin"}'
""").collect()

df = session.sql("""
    SELECT TENANT_ID, COUNT(*) AS ROW_COUNT, SUM(AMOUNT) AS TOTAL_REVENUE
    FROM SALES_DATA
    GROUP BY TENANT_ID
""").to_pandas()
print("=== As globex_inc: Only globex_inc rows visible ===")
print(df.to_string(index=False))
print(f"\nTotal rows visible: {df['ROW_COUNT'].sum()}")

In [ ]:
# Reset session attributes
session.sql("ALTER SESSION UNSET SNOWFLAKE$SESSION_ATTRIBUTES").collect()
print("Session attributes reset.")

# Confirm all data is visible again (owner role bypasses RAP)
count = session.sql("SELECT COUNT(*) AS CNT FROM SALES_DATA").collect()[0]['CNT']
print(f"Total rows visible after reset: {count}")

---
## Section 5: Test Column Masking Manually

Column masking is based on `access_level` (looked up via the `GET_USER_ACCESS_LEVEL()` UDF):
- **full** — sees everything (names + emails)
- **summary** — sees names, emails masked
- **restricted** — both names and emails masked

In [ ]:
# Test as a RESTRICTED user — both name and email should be masked
session.sql("""
    ALTER SESSION SET SNOWFLAKE$SESSION_ATTRIBUTES = 
    '{"tenant_id": "acme_corp", "user_id": "user_acme_viewer"}'
""").collect()

df = session.sql("""
    SELECT SALE_ID, PRODUCT, AMOUNT, CUSTOMER_NAME, CUSTOMER_EMAIL
    FROM SALES_DATA
    LIMIT 5
""").to_pandas()
print("=== access_level='restricted' (user_acme_viewer) ===")
print("Expected: CUSTOMER_NAME and CUSTOMER_EMAIL both masked")
print(df.to_string(index=False))

In [ ]:
# Test as a FULL access user — everything visible
session.sql("""
    ALTER SESSION SET SNOWFLAKE$SESSION_ATTRIBUTES = 
    '{"tenant_id": "acme_corp", "user_id": "user_acme_admin"}'
""").collect()

df = session.sql("""
    SELECT SALE_ID, PRODUCT, AMOUNT, CUSTOMER_NAME, CUSTOMER_EMAIL
    FROM SALES_DATA
    LIMIT 5
""").to_pandas()
print("=== access_level='full' (user_acme_admin) ===")
print("Expected: CUSTOMER_NAME and CUSTOMER_EMAIL fully visible")
print(df.to_string(index=False))

In [ ]:
# Test as a SUMMARY user — name visible, email masked
session.sql("""
    ALTER SESSION SET SNOWFLAKE$SESSION_ATTRIBUTES = 
    '{"tenant_id": "acme_corp", "user_id": "user_acme_analyst1"}'
""").collect()

df = session.sql("""
    SELECT SALE_ID, PRODUCT, AMOUNT, CUSTOMER_NAME, CUSTOMER_EMAIL
    FROM SALES_DATA
    LIMIT 5
""").to_pandas()
print("=== access_level='summary' (user_acme_analyst1) ===")
print("Expected: CUSTOMER_NAME visible, CUSTOMER_EMAIL masked")
print(df.to_string(index=False))

In [ ]:
# Reset for next section
session.sql("ALTER SESSION UNSET SNOWFLAKE$SESSION_ATTRIBUTES").collect()
print("Session attributes reset.")

---
## Section 6: Call the Agent with Tenant Context

Now the real magic: we call `SNOWFLAKE.CORTEX.AGENT()` with `variables` that set the session context. The agent's generated SQL runs *within* the policy-enforced session — it automatically only sees the calling tenant's data.

Same question, different tenants, different answers.

In [ ]:
import json

def call_agent(question, tenant_id, user_id):
    """Call TENANT_SALES_AGENT with tenant context via variables."""
    variables = json.dumps({"tenant_id": tenant_id, "user_id": user_id})
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.AGENT(
            'MULTI_TENANCY_LAB.PUBLIC.TENANT_SALES_AGENT',
            '{question}',
            PARSE_JSON('{variables}')
        ) AS response
    """).collect()
    return result[0]['RESPONSE']

print("Agent helper function defined.")

In [ ]:
# Ask acme_corp about their total revenue
response = call_agent(
    "What was our total revenue in 2024?",
    tenant_id="acme_corp",
    user_id="user_acme_admin"
)
print("=== Agent Response (tenant: acme_corp) ===")
print(response)

In [ ]:
# Same question for globex_inc — different answer!
response = call_agent(
    "What was our total revenue in 2024?",
    tenant_id="globex_inc",
    user_id="user_globex_admin"
)
print("=== Agent Response (tenant: globex_inc) ===")
print(response)

In [ ]:
# Verify isolation: ask about a specific product only one tenant has
response = call_agent(
    "How many orders did we have for Biotech Analytics Suite?",
    tenant_id="acme_corp",
    user_id="user_acme_admin"
)
print("=== acme_corp asking about umbrella_co's product ===")
print("Expected: 0 results or 'no data found' (acme_corp can't see umbrella_co data)")
print(response)

In [ ]:
# Same product question from the correct tenant
response = call_agent(
    "How many orders did we have for Biotech Analytics Suite?",
    tenant_id="umbrella_co",
    user_id="user_umbrella_exec"
)
print("=== umbrella_co asking about their own product ===")
print("Expected: Results showing Biotech Analytics Suite orders")
print(response)

---
## Section 7: Column Masking Through the Agent

The masking policies apply to the agent's generated SQL too. When a user with `restricted` access asks for customer details, the agent returns masked values — it doesn't even know the real data exists.

In [ ]:
# Restricted user asking for customer details
response = call_agent(
    "Show me the top 5 customers by revenue.",
    tenant_id="acme_corp",
    user_id="user_acme_viewer"  # restricted access
)
print("=== Restricted user: customer names and emails masked ===")
print(response)

In [ ]:
# Full access user — same question, full details visible
response = call_agent(
    "Show me the top 5 customers by revenue.",
    tenant_id="acme_corp",
    user_id="user_acme_admin"  # full access
)
print("=== Full access user: customer names and emails visible ===")
print(response)

In [ ]:
# Summary user — names visible, emails masked
response = call_agent(
    "Show me the top 5 customers by revenue with their email addresses.",
    tenant_id="globex_inc",
    user_id="user_globex_analyst"  # summary access
)
print("=== Summary user: names visible, emails masked ===")
print(response)

---
## Section 8: Add a New User (No DDL!)

One of the biggest advantages of this approach: adding a new external user requires **zero DDL changes**. No new roles, no GRANT statements, no policy modifications. Just INSERT a row into the entitlements table.

In [ ]:
# Add a brand new user for initech — a new hire with summary access
session.sql("""
    INSERT INTO USER_ENTITLEMENTS VALUES
    ('user_initech_newhire', 'initech', 'summary')
""").collect()
print("New user 'user_initech_newhire' added with summary access to initech.")

# Verify the insert
df = session.sql("""
    SELECT * FROM USER_ENTITLEMENTS WHERE EXTERNAL_USER_ID = 'user_initech_newhire'
""").to_pandas()
print(df.to_string(index=False))

In [ ]:
# Immediately call the agent as the new user — it just works!
response = call_agent(
    "What was our total revenue by product in 2024?",
    tenant_id="initech",
    user_id="user_initech_newhire"
)
print("=== New user (user_initech_newhire) — instant access! ===")
print("No role grants. No policy changes. Just an INSERT.")
print()
print(response)

In [ ]:
# Verify masking is correct for the new user (summary = name visible, email masked)
response = call_agent(
    "Show me the 3 most recent sales with customer details.",
    tenant_id="initech",
    user_id="user_initech_newhire"  # summary access
)
print("=== New user masking check (summary level) ===")
print("Expected: Customer names visible, emails masked")
print()
print(response)

---
## Section 9: Audit Trail

Every agent invocation is logged in `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY`. This gives you a complete audit trail of who asked what, with the ability to correlate requests back to tenant context.

In [ ]:
# Query the agent usage history
# Note: ACCOUNT_USAGE views have up to 45-minute latency
df = session.sql("""
    SELECT 
        REQUEST_ID,
        AGENT_NAME,
        USER_QUERY,
        START_TIME,
        CREDITS_USED
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
    WHERE AGENT_NAME = 'TENANT_SALES_AGENT'
      AND START_TIME > DATEADD('hour', -2, CURRENT_TIMESTAMP())
    ORDER BY START_TIME DESC
    LIMIT 10
""").to_pandas()

if len(df) > 0:
    print("=== Recent Agent Requests ===")
    print(df.to_string(index=False))
else:
    print("No usage history yet (ACCOUNT_USAGE views have up to 45-min latency).")
    print("Re-run this cell later to see your requests logged.")

---
## Section 10: Summary

### What You Accomplished

| Step | What you did |
|------|-------------|
| **Data Model** | Explored a multi-tenant sales dataset with 120 rows across 4 companies |
| **Row Access Policy** | Tested automatic tenant isolation via `SYS_CONTEXT` session attributes |
| **Column Masking** | Verified 3-tier masking (full/summary/restricted) based on user entitlements |
| **Agent + RAP** | Called the Cortex Agent with tenant context and saw automatic row filtering |
| **Agent + Masking** | Observed column masking applied transparently through agent responses |
| **Zero-DDL Onboarding** | Added a new user with a single INSERT — instant access, no grants needed |
| **Audit Trail** | Queried usage history for request attribution and compliance |

### Key Takeaways

1. **No Snowflake accounts needed** — External users are identified by session attributes, not Snowflake users/roles
2. **Policies are invisible to the agent** — The LLM generates SQL normally; policies filter/mask automatically
3. **Onboarding is O(1)** — Adding a new tenant or user is a DML operation, not a DDL operation
4. **Audit is built-in** — Every request is logged with correlation IDs for compliance

### What's Next

- **Integrate with your IdP** — Map SAML/OIDC claims to session attributes via your application layer
- **Add Cortex Search** — Create per-tenant search services for document RAG with the same isolation
- **Set up resource budgets** — Use Snowflake budgets to track and limit per-tenant compute costs
- **Add more entitlement dimensions** — Extend the model with region-level access, time-based restrictions, etc.
- **Production hardening** — Add error handling, rate limiting, and monitoring dashboards

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DELETE FROM USER_ENTITLEMENTS WHERE EXTERNAL_USER_ID = 'user_initech_newhire'").collect()
# session.sql("DROP DATABASE IF EXISTS MULTI_TENANCY_LAB CASCADE").collect()
# print("Lab resources cleaned up.")